In [16]:
#configuration
from pathlib import Path

ROOT_DIR = Path.cwd().parent.parent

DATASET_PATH = ROOT_DIR / "eval" / "retrieval_evaluation" / "data" / "retrieval_eval.json"
DOC_STORE_PATH = ROOT_DIR / "data" / "faiss_storage" / "doc_store.json"
CONTEXTS_PATH = ROOT_DIR / "database" / "contexts.json"

RESULTS_DIR = ROOT_DIR / "retrieval_evaluation" / "results"

print("ROOT_DIR:", ROOT_DIR)
print("DATASET:", DATASET_PATH, DATASET_PATH.exists())
print("DOC_STORE:", DOC_STORE_PATH, DOC_STORE_PATH.exists())
print("CONTEXTS:", CONTEXTS_PATH, CONTEXTS_PATH.exists())
print("RESULTS:", RESULTS_DIR)
K_VALUES = [1, 3, 5, 10, 20]

ROOT_DIR: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit
DATASET: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/data/retrieval_eval.json True
DOC_STORE: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/data/faiss_storage/doc_store.json True
CONTEXTS: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/database/contexts.json True
RESULTS: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/retrieval_evaluation/results


In [27]:
#load dataset
import json

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Số lượng queries:", len(eval_data))
print("Query đầu tiên:")
print(eval_data[0])

Số lượng queries: 182
Query đầu tiên:
{'id': 'd55bbab59cdd93ee833f82d41b03669486dfc8f592e931dd572dcdcd0edbbb59', 'query': 'Mùng 1 âm lịch có tên gọi là gì?', 'relevant_documents': [{'document_id': '8cd35d91-d78c-4883-b8d6-ceea61bc3a14', 'relevance': 2}]}


In [23]:
#canonical relevance
def get_relevance_map(record):
    return {
        item["document_id"]: item["relevance"]
        for item in record["relevant_documents"]
    }

In [38]:
import sys
from pathlib import Path

SRC_PATH = ROOT_DIR / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from rag.retriever import Retriever

print("Retriever imported successfully!")

/home/sotatek/Projects/Tools/Notebook/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retriever imported successfully!


In [40]:
retriever = Retriever(
    index_path=str(INDEX_PATH),
    contexts_path=str(CONTEXTS_PATH)
)

print("Retriever initialized successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19406.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever initialized successfully!


In [24]:
#adapter
#Vì FAISS index khác document_id nên phải tạo 1 adapter để chuyển từ FAISS index -> content_hash -> document_id
def adapt_retrieval_results(raw_result, faiss_idx_to_hash, hash_to_document_id):
    adapted = []

    for doc in raw_result["documents"]:
        faiss_id = int(doc["id"].replace("doc_", ""))

        content_hash = faiss_idx_to_hash[faiss_id]

        document_id = hash_to_document_id[content_hash]

        adapted.append({
            "document_id": document_id,
            "score": doc["metadata"]["retrieval_score"]
        })

    return adapted